<a href="https://colab.research.google.com/github/100522155/egci463/blob/main/digit_pca_perceptron_pipeline_gridsearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Digits: PCA + Perceptron in a Pipeline with GridSearch

Exercise notebook:
- Load a local `digit.csv` file (expects features + a label column).
- Apply PCA as dimensionality reduction.
- Train a Perceptron classifier.
- Use Pipeline and  GridSearchCV to find best hyperparameters.
- Evaluate on a held-out test set and show a few visualizations (explained variance, 2D PCA projection, confusion matrix).


In [ ]:
# Basic imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# 1) Load digit.csv (assumes file is in the notebook working directory)
df = pd.read_csv('digit.csv',header=None)
print('Loaded shape:', df.shape)
df.head()

Loaded shape: (500, 785)


,0,1,2,3,4,5,6,7,8,9,...,775,776,777,778,779,780,781,782,783,784
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Split the data into train / test (hold out test set as a locked box). We will run GridSearchCV on the training portion (For validation).

In [ ]:
# 3) Train/Test split
from sklearn.model_selection import train_test_split

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

Build a pipeline: StandardScaler -> PCA -> Perceptron. We keep PCA as a named step so we can grid-search n_components and later inspect explained variance.

In [ ]:
# 4) Pipeline definition


from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Perceptron
steps = [
    ('scaler', StandardScaler()),
    ('pca', PCA(random_state=42)),
    ('clf', Perceptron(random_state=42))
]

pipeline = Pipeline(steps)

Set up a small parameter grid  
- pca__n_components: try a few dimensionalities (Remember the real size is 28*28)
- Use  few pameter from clssfiction


In [ ]:
# 5) Parameter grid

param_grid = {
    'pca__n_components': [10, 20, 30, 50, 100],
    'clf__eta0': [0.0001, 0.001, 0.01, 0.1],
    'clf__max_iter': [500, 1000],
    'clf__penalty': [None, 'l2', 'l1'],
}
param_grid

Run GridSearchCV with StratifiedKFold/or CrossValidationKfold (fold 4-7 of your chose) on the training data. The search will fit preprocessing inside each fold because we used a Pipeline.

In [ ]:
# 6) Grid search

from sklearn.model_selection import GridSearchCV, StratifiedKFold

# GridSearch with Cross-Validation of 5 folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gs = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Adjust train group
gs.fit(X_train, y_train)

print("Best Parameters:", gs.best_params_)
print("Best CV Accuracy:", gs.best_score_)

Inspect a small summary of cv_results_.

In [ ]:
# 7) Inspect results (compact)
import pandas as pd
results = pd.DataFrame(gs.cv_results_)
cols = ['param_pca__n_components', 'param_clf__eta0',  'mean_test_score', 'std_test_score', 'rank_test_score']# add more parameter as needed
display(results[cols].sort_values('rank_test_score').head(10))


Evaluate the best estimator on the held-out test set (one-time evaluation).

In [ ]:
# 8) Final evaluation on test set
# Best Estimator
# Test on the test set
#Show confusion matrix
#Plot the loss value
#ROC is
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# OBest estimator
best = gs.best_estimator_

# test set evaluation
y_pred = best.predict(X_test)

print("--- Test Set Classification Report ---")
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix (Test Set)")
plt.show()

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# 1) Convert the labels (0-9) to "one-vs-rest" format:
#    one column per digit, 1 if the image is that digit, 0 otherwise
y_bin = label_binarize(y_test, classes=range(10))

# 2) Model scores for each digit
#    (Perceptron has no probabilities, so we use decision_function)
scores = best.decision_function(X_test)

# 3) Draw one ROC curve per digit
plt.figure(figsize=(7, 6))
for d in range(10):
    fpr, tpr, _ = roc_curve(y_bin[:, d], scores[:, d])   # false / true positive rates
    plt.plot(fpr, tpr, label=f'digit {d} (AUC={auc(fpr, tpr):.2f})')

plt.plot([0, 1], [0, 1], 'k--')   # diagonal line = random guessing
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve (one-vs-rest)')
plt.legend(fontsize=8)
plt.show()

Examine the PCA explained variance for the selected PCA step. The pipeline was refit on the full training data by GridSearchCV (refit=True), so we can inspect the PCA inside the best estimator.

In [ ]:
# 9) PCA explained variance (from best estimator)
best_pca = best.named_steps['pca']
explained = best_pca.explained_variance_ratio_
cumvar = np.cumsum(explained)
plt.figure(figsize=(6,4))
plt.plot(np.arange(1, len(explained)+1), cumvar, marker='o')
plt.xlabel('Principal component index')
plt.ylabel('Cumulative explained variance')
plt.title('Cumulative explained variance (best PCA)')
plt.grid(True)
plt.show()

print('Number of components used by best PCA:', best_pca.n_components_)


In [ ]:
#Plot the loss function
#name the best parameter and consider the fit
from sklearn.linear_model import Perceptron

# 1) Get the best parameters found by GridSearch
bp = gs.best_params_

# 2) Apply the fitted preprocessing (scaler + PCA) to the data
#    best[:-1] means "every step of the pipeline except the last one (the Perceptron)"
Z_train = best[:-1].transform(X_train)
Z_test = best[:-1].transform(X_test)

# 3) Create a new Perceptron with the best parameters
clf = Perceptron(eta0=bp['clf__eta0'], penalty=bp['clf__penalty'], random_state=42)

# 4) Train it one epoch at a time and record the errors after each epoch
train_loss, test_loss = [], []
for epoch in range(100):
    clf.partial_fit(Z_train, y_train, classes=np.arange(10))   # one pass over the data
    train_loss.append(1 - clf.score(Z_train, y_train))         # train error = 1 - accuracy
    test_loss.append(1 - clf.score(Z_test, y_test))            # test error  = 1 - accuracy

# 5) Plot the loss curve
plt.figure(figsize=(7, 4))
plt.plot(train_loss, label='train error')
plt.plot(test_loss, label='test error')
plt.xlabel('Epoch')
plt.ylabel('Error rate (loss)')
plt.title('Perceptron loss per epoch')
plt.legend()
plt.grid(True)
plt.show()

# 6) Name the best parameters and check the fit
print('Best parameters:', gs.best_params_)
print('CV accuracy   :', round(gs.best_score_, 4))
print('Train accuracy:', round(best.score(X_train, y_train), 4))
print('Test accuracy :', round(best.score(X_test, y_test), 4))
#pipe.fit(X, y)


Visualize a 2D PCA projection of the test set (using the scaler + PCA steps from the fitted pipeline). This helps see class separation in reduced space.

In [ ]:
# 10) 2D projection (if PCA has >=2 components)
scaler = best.named_steps['scaler']
pca = best.named_steps['pca']

X_test_scaled = scaler.transform(X_test)
X_test_pca = pca.transform(X_test_scaled)

if X_test_pca.shape[1] >= 2:
    plt.figure(figsize=(8,6))
    scatter = plt.scatter(X_test_pca[:, 0], X_test_pca[:, 1], c=y_test, cmap='tab10', alpha=0.7)
    plt.colorbar(scatter, label='Digit Class')
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.title('2D PCA Projection of Test Data')
    plt.grid(True)
    plt.show()


-Try remove PCA step and detect the best

In [ ]:
# Same parameters as before, but there is no PCA step, so no "pca__" parameter
param_grid_nopca = {
    'clf__eta0': [0.0001, 0.001, 0.01, 0.1],
    'clf__max_iter': [500, 1000],
    'clf__penalty': [None, 'l2', 'l1'],
}

# Option A: Scaler -> Perceptron (no PCA)
pipe_a = Pipeline([('scaler', StandardScaler()),
                   ('clf', Perceptron(random_state=42))])
gs_a = GridSearchCV(pipe_a, param_grid_nopca, cv=cv, scoring='accuracy', n_jobs=-1)
gs_a.fit(X_train, y_train)

# Option B: Perceptron only (no scaler, no PCA)
pipe_b = Pipeline([('clf', Perceptron(random_state=42))])
gs_b = GridSearchCV(pipe_b, param_grid_nopca, cv=cv, scoring='accuracy', n_jobs=-1)
gs_b.fit(X_train, y_train)

# Compare the 3 models: CV accuracy (on train data) and test accuracy
models = {
    'Scaler + PCA + Perceptron': gs,     # the model from the main grid search
    'Scaler + Perceptron':       gs_a,
    'Perceptron only':           gs_b,
}

for name, g in models.items():
    print(name)
    print('  best params :', g.best_params_)
    print('  CV accuracy :', round(g.best_score_, 4))
    print('  test accuracy:', round(g.best_estimator_.score(X_test, y_test), 4))
    print()

Quick Note.....

## Quick Note

**Best model (Scaler + PCA + Perceptron):** pca__n_components = 50, eta0 = 0.1, max_iter = 500, penalty = 'l2'.
CV accuracy = 0.78, test accuracy = 0.76.

**Fit:** train accuracy (0.91) is much higher than test accuracy (0.76), so the model overfits.
This is expected: only 400 training images and 784 pixel features.

**Errors:** the confusion matrix shows digits 4 and 8 are the hardest (recall 0.40).
The ROC curves are good (average AUC about 0.93), so the model ranks the right digit well even when the final prediction is wrong.

**Loss curve:** the error does not decrease smoothly. A Perceptron only updates its weights when it makes a mistake, so the curve is noisy.

**PCA vs. no PCA:** the Perceptron alone (no scaler, no PCA) gave the best result
(CV 0.835, test 0.88). StandardScaler gives the empty border pixels the same weight as the
important pixels, which adds noise. PCA reduced 784 features to 50 and kept about 76% of the variance,
which made training faster but lost some information.

**Conclusion:** a Perceptron is a linear model, so it is limited on digit images.
A non-linear model such as an MLP (next week) should do better.